## Moving Averages Strategy

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


In [ ]:
# Get data
data = pd.read_csv('../data/raw_ohlcv/BTCUSDT-1h-2017-08-17.csv', parse_dates=['date'], index_col='date')
data = data[['close', 'volume']]
data = data.loc['2021'].copy()

In [ ]:
# Buy and hold benchmark
data['return'] = data['close'].div(data['close'].shift(1))
data['return'] = np.log(data['return'])
data['c_return'] = np.exp(data['return'].cumsum())

#### Simple Moving Average Strategy
This is just a simple long-only strategy that buys when the lower timeframe MA moves above the higher timeframe MA, and sells again once it does the opposite. This is just very basic logic to start - ideally, there should be independent sell conditions, rather than just selling when the trend reverses, otherwise the strategy is just lagging. The sanity just see that there's a reasonable number of times in and out of a trade (would expect about equal since it's based on above/below and average)

In [ ]:
# Moving average strategy
data['position'] = 0

MA_S = 50
MA_L = 100

data['ma_s'] = data['close'].rolling(window=MA_S).mean()
data['ma_l'] = data['close'].rolling(window=MA_L).mean()

data['condition'] = data['ma_s'] > data['ma_l']

data.loc[data['condition'], 'position'] = 1

# Sanity check
data['position'].value_counts()


In [ ]:
# Seeing where trades occur
data['trade'] = data['position'].diff()

fig, myax = plt.subplots(figsize=(12,8))    # Create the figure and axis to plot onto

data[['close', 'ma_s', 'ma_l']].loc['2021-01':'2021-02'].plot(ax=myax)

buy_times = data.loc[data['trade'] == 1]
sell_times = data.loc[data['trade'] == -1]

buy_times = buy_times.loc['2021-01':'2021-02'].index
sell_times = sell_times.loc['2021-01':'2021-02'].index

ymin, ymax = myax.get_ylim()

myax.vlines(buy_times, ymin, ymax, color='green', linestyle='dotted')
myax.vlines(sell_times, ymin, ymax, color='red', linestyle='dashed')

plt.show()